In [ ]:
import os
import time
import pandas as pd
from sqlalchemy import create_engine
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ---------------------------------------------------
# 1. Verbindung zur PostgreSQL-Datenbank
# ---------------------------------------------------

DB_USER = os.getenv("DB_USER", "sbs")
DB_PASS = os.getenv("DB_PASS", "qaqpav-xyxhi9-jeGmyv")
DB_HOST = os.getenv("DB_HOST", "89.63.6.173")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "wenke")

print("Verbinde mit PostgreSQL...")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Verbindung erfolgreich.")

# ---------------------------------------------------
# 2. Daten laden
# ---------------------------------------------------

query = """
SELECT
    adr_adressnummer,
    adr_adresse as adresse,
    adr_plz as plz,
    adr_stadt as stadt
FROM reporting.silver_adresse
"""

print("Lade Adressen aus Datenbank...")

df = pd.read_sql(query, engine)

print(f"{len(df)} Adressen geladen.")

# ---------------------------------------------------
# 3. Vollständige Adresse bauen
# ---------------------------------------------------

df["full_address"] = (
    df["adresse"].astype(str).str.strip() + ", " +
    df["plz"].astype(str).str.strip() + " " +
    df["stadt"].astype(str).str.strip() +
    ", Deutschland"
)

# ---------------------------------------------------
# 4. Geocoder vorbereiten
# ---------------------------------------------------

print("Initialisiere Geocoder...")

geolocator = Nominatim(user_agent="postgres_geocoder")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1
)

# ---------------------------------------------------
# 5. Koordinaten holen
# ---------------------------------------------------

success_count = 0
error_count = 0
start_time = time.time()

def get_coordinates(row):
    global success_count, error_count

    index = row.name + 1
    total = len(df)
    address = row["full_address"]

    print(f"[{index}/{total}] Geocodiere: {address}")

    try:
        location = geocode(address)

        if location:
            success_count += 1

            elapsed = time.time() - start_time
            avg_time = elapsed / index
            remaining = avg_time * (total - index)

            print(
                f"   OK -> "
                f"Lat: {location.latitude:.6f}, "
                f"Lon: {location.longitude:.6f} | "
                f"Erfolgreich: {success_count} | "
                f"Fehler: {error_count} | "
                f"Restzeit ca.: {remaining/60:.1f} min"
            )

            return pd.Series({
                "latitude": location.latitude,
                "longitude": location.longitude
            })

        else:
            error_count += 1
            print(f"   NICHT GEFUNDEN")

    except Exception as e:
        error_count += 1
        print(f"   FEHLER bei {address}: {e}")

    return pd.Series({
        "latitude": None,
        "longitude": None
    })

df[["latitude", "longitude"]] = df.apply(get_coordinates, axis=1)

# ---------------------------------------------------
# 6. Ergebnis anzeigen
# ---------------------------------------------------

print("\nGeocoding abgeschlossen.")
print(f"Erfolgreich: {success_count}")
print(f"Fehlerhaft: {error_count}")

print(df.head())

# ---------------------------------------------------
# 7. Zurück in PostgreSQL speichern
# ---------------------------------------------------

print("Speichere Daten in PostgreSQL...")

df.to_sql(
    "silver_adresse_geocoded",
    engine,
    schema="reporting",
    if_exists="replace",
    index=False
)

print("Tabelle reporting.silver_adresse_geocoded gespeichert.")

In [ ]:
from sqlalchemy import create_engine

df.to_sql(
    "raw_adresse_geocoded",
    engine,
    schema="reporting",
    if_exists="replace",
    index=False
)

In [ ]:
df.to_csv(
    "raw_adresse_geocoded.csv",
    index=False,
    encoding="utf-8"
)